# Iterators, Generators & Context Managers

*Iterator protocol · yield · yield from · send() · itertools (complete) · @contextmanager · ExitStack · Real-World*


---
## Introduction


# Iterators Generators Decorators

*Run each cell with **Shift+Enter***

00 — Python Foundations: Iterators, Generators, Decorators
==========================================================

Runnable companion to PDF Chapter "P+ — Python Foundations"
(iterators/generators/decorators sections).

  * Iterator protocol (__iter__/__next__)
  * Generators (yield) and lazy pipelines
  * Generator expressions
  * Decorators (timing, memoization, retry) with functools.wraps

Run:  python iterators_generators_decorators.py


---
## 🧠 Notebook Mental Model: Iterators, Generators & Context Managers

> **Think of an iterator as a cursor that moves forward through data — one step at a time, on demand.**  
> A generator is a function that *pauses* at each `yield` and *resumes* when you ask for the next value.

### The Lazy Evaluation Stack

```
CONSUMER (for loop, sum, list(), …)
        │  calls next()
        ▼
 GENERATOR / ITERATOR  ←── state is FROZEN here between calls
        │  calls next() on sub-source
        ▼
 DATA SOURCE  (file, network, list, infinite sequence, …)

 Benefit: only one item in memory at a time — constant O(1) space.
```

### Why / What / How / When at a Glance

| Concept | WHY | WHAT | HOW | WHEN |
|---------|-----|------|-----|------|
| **Iterator protocol** | Uniform access to any data source | Object with `__iter__` + `__next__` | `__next__` returns one item or raises `StopIteration` | Implementing custom lazy data sources |
| **Generator function** | Easy iterator creation | Function with `yield` | Execution pauses at `yield`; resumes on `next()` | Any lazy sequence |
| **Generator expression** | One-liner generators | `(expr for x in it)` | Same as generator function but compact | Feeding into aggregators |
| **`yield from`** | Delegate to sub-generator | Transparent forwarding | Passes `send()`/`throw()` through | Composing pipelines, flattening |
| **`.send(val)`** | Two-way communication | Push a value INTO a generator | Becomes the value of the `yield` expression | Coroutines, data injection |
| **`itertools`** | Production-grade lazy combinators | C-speed lazy operations | See table below | Data pipelines, combinations |
| **Context manager** | Guaranteed resource cleanup | `with` statement | `__enter__` / `__exit__` protocol | Files, DB connections, locks |

### The `yield` Execution Model

```
def gen():
    print("start")
    x = yield 1         ←── PAUSES here; "1" is sent to caller
    print(f"resumed with {x}")
    yield 2             ←── PAUSES again

g = gen()
v = next(g)            → prints "start", v = 1
g.send("hello")        → prints "resumed with hello", returns 2
```

### `itertools` Cheat Sheet (most important)

| Function | What | When |
|----------|------|------|
| `count(n)` | Infinite counter | Numbering without range |
| `cycle(it)` | Infinite round-robin | Load balancer, rotating headers |
| `islice(it, n)` | Take first n lazily | Limit infinite generators |
| `chain(*its)` | Concatenate iterables lazily | Flatten one level |
| `product(a, b)` | Cartesian product | Grid search, combinations |
| `combinations(it, r)` | C(n,r) without repetition | Pair selection |
| `groupby(it, key)` | Group consecutive runs | Must sort first! |
| `takewhile(pred, it)` | Take while predicate holds | Prefix of a stream |
| `dropwhile(pred, it)` | Skip while predicate holds | Skip header lines |
| `accumulate(it, fn)` | Running totals | Prefix sums, max-so-far |


In [ ]:
from __future__ import annotations

import functools
import time
from collections.abc import Iterator

===========================================================================
ITERATOR PROTOCOL — the machinery behind every `for` loop
===========================================================================


### 🧠 Mental Model: Iterator Protocol

**WHY** — Every `for` loop in Python secretly uses the iterator protocol. Understanding it lets you make any object work in a `for` loop, `list()`, `sum()`, `zip()`, etc.

**WHAT** — An iterable has `__iter__()`. An iterator has both `__iter__()` and `__next__()`. The `for` loop calls `iter()` on the iterable, then `next()` repeatedly.

**HOW — the for loop desugared:**
```
for x in obj:         →   _it = iter(obj)    # calls obj.__iter__()
    body                   while True:
                               try:
                                   x = next(_it)  # calls _it.__next__()
                                   body
                               except StopIteration:
                                   break
```

**Iterable vs Iterator — the key distinction:**
```
ITERABLE  — can produce an iterator; can be iterated MULTIPLE times
  Examples: list, tuple, str, dict, set, range

ITERATOR  — stateful cursor; can only be traversed ONCE (raises StopIteration when exhausted)
  Examples: file objects, zip(), map(), filter(), generator functions

Pythonic idiom: isinstance(x, Iterator) to check
```

**WHEN to implement your own iterator:**
- Custom data structures (tree, graph traversal)
- Reading from an external source (DB cursor, socket chunks)
- Lazy transformation pipelines
- When you need `__iter__` on a class that also IS an iterator (make `__iter__` return `self`)


In [ ]:
class Countdown:
    """A hand-written iterator: implements __iter__ and __next__."""

    def __init__(self, start: int) -> None:
        self.current = start

    def __iter__(self) -> Countdown:
        return self

    def __next__(self) -> int:
        if self.current <= 0:
            raise StopIteration
        self.current -= 1
        return self.current + 1

===========================================================================
GENERATORS — the easy way to make an iterator (lazy, memory-cheap)
===========================================================================


### 🧠 Mental Model: Generators

**WHY** — A generator is the simplest way to create an iterator. It avoids storing all results in memory and can represent infinite sequences.

**WHAT** — A generator function contains `yield`. When called, it returns a generator object immediately without running any code. Code runs lazily as you call `next()`.

**HOW — state machine diagram:**
```
generator = my_gen()     ← function body NOT executed yet

next(generator)          ← runs until first yield, pauses, returns value
next(generator)          ← resumes from after yield, runs to next yield
next(generator)          ← raises StopIteration when function returns/exits

State: [ NOT_STARTED → RUNNING ⇄ SUSPENDED → CLOSED ]
```

**`yield from` — delegation:**
```python
def flatten(nested):
    for sublist in nested:
        yield from sublist   # ← expands to: for x in sublist: yield x
        # BUT also forwards .send(), .throw(), .close() to the sub-generator

# Equivalent without yield from:
def flatten_manual(nested):
    for sublist in nested:
        for x in sublist:
            yield x
```

**Memory comparison:**
```python
# Eager: builds full list first — uses O(n) memory
def squares_list(n):
    return [i*i for i in range(n)]

# Lazy: computes one at a time — O(1) memory regardless of n
def squares_gen(n):
    for i in range(n):
        yield i * i

# For n=10^8, the list takes ~800 MB; the generator takes ~200 bytes
```

**WHEN to use generators:**
- Processing large files line by line (`yield line`)
- Implementing infinite sequences (Fibonacci, primes)
- Building data pipelines (chain generators together)
- Replacing callbacks with coroutines (using `.send()`)


In [ ]:
def countdown(n: int) -> Iterator[int]:
    while n > 0:
        yield n  # produces one value at a time; state is suspended between calls
        n -= 1


def fibonacci() -> Iterator[int]:
    """An INFINITE generator — impossible with a list."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


def take(gen: Iterator[int], k: int) -> list[int]:
    return [next(gen) for _ in range(k)]

===========================================================================
DECORATORS — add behavior without editing the wrapped function
===========================================================================

In [ ]:
def timed(fn):
    @functools.wraps(fn)  # preserve name/docstring of the wrapped function
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = fn(*args, **kwargs)
        wrapper.last_seconds = time.perf_counter() - start  # type: ignore[attr-defined]
        return result

    wrapper.last_seconds = 0.0  # type: ignore[attr-defined]
    return wrapper


def retry(times: int = 3):
    """A decorator FACTORY — parameterized decorator."""

    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last_exc: Exception | None = None
            for _ in range(times):
                try:
                    return fn(*args, **kwargs)
                except Exception as exc:  # noqa: BLE001 (demo)
                    last_exc = exc
            raise last_exc  # type: ignore[misc]

        return wrapper

    return decorator


def memoize(fn):
    cache: dict[tuple, object] = {}

    @functools.wraps(fn)
    def wrapper(*args):
        if args not in cache:
            cache[args] = fn(*args)
        return cache[args]

    wrapper.cache = cache  # type: ignore[attr-defined]
    return wrapper


@memoize
def slow_square(n: int) -> int:
    return n * n


@timed
def sum_to(n: int) -> int:
    return sum(range(n))


_flaky_calls = {"n": 0}


@retry(times=3)
def flaky() -> str:
    _flaky_calls["n"] += 1
    if _flaky_calls["n"] < 3:
        raise ConnectionError("transient")
    return "ok"


def main() -> None:
    print("=" * 68)
    print("PYTHON FOUNDATIONS — iterators_generators_decorators.py")
    print("=" * 68)

    assert list(Countdown(3)) == [3, 2, 1]
    assert list(countdown(3)) == [3, 2, 1]
    print("iterators/generators: countdown ->", list(countdown(3)))

    # Infinite generator, consumed lazily.
    assert take(fibonacci(), 8) == [0, 1, 1, 2, 3, 5, 8, 13]
    print("infinite generator: first 8 fibs ->", take(fibonacci(), 8))

    # Generator expression — lazy, no giant list materialized.
    total = sum(n * n for n in range(1_000))
    assert total == 332_833_500
    print("generator expression: sum of squares < 1000 ->", total)

    # Decorators
    assert sum_to(1_000) == 499_500
    print(f"@timed: sum_to(1000) ran in {sum_to.last_seconds:.6f}s")

    assert slow_square(12) == 144 and slow_square(12) == 144
    assert (12,) in slow_square.cache
    print("@memoize: 12^2 cached ->", slow_square.cache)

    assert flaky() == "ok" and _flaky_calls["n"] == 3
    print("@retry: succeeded on attempt", _flaky_calls["n"])

    print("-" * 68)
    print("All iterator/generator/decorator demos passed ✔")


if __name__ == "__main__":
    # Keep Unicode output safe even when stdout is redirected/piped (Windows cp1252 fallback).
    import sys
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()

---
## Iterators, Generators & Context Managers — Complete


# Iterators Generators Context Complete

*Run each cell with **Shift+Enter***

Iterators, Generators, Context Managers & itertools — Complete Reference
========================================================================
Mental model:
  Iterator  = a CURSOR that walks a sequence one step at a time.
  Generator = a function that SUSPENDS its stack frame between yields.
  Context Manager = a CLEANUP GUARANTEE — setup runs on enter, teardown on exit.

PART 1 : Iterator protocol      — __iter__, __next__, StopIteration
PART 2 : Generator functions    — yield, lazy evaluation, infinite sequences
PART 3 : Generator expressions  — lazy comprehension equivalent
PART 4 : yield from             — delegation and transparent forwarding
PART 5 : send() / throw()       — two-way generator communication
PART 6 : itertools              — lazy combinatorial library (complete)
PART 7 : Context managers       — __enter__/__exit__ class protocol
PART 8 : @contextmanager        — generator-based context managers
PART 9 : contextlib utilities   — suppress, nullcontext, ExitStack
PART 10: Async generators       — async for, async with (preview)

Run: python iterators_generators_context_complete.py

In [ ]:
from __future__ import annotations

import sys
import time
import itertools
from contextlib import (
    contextmanager,
    suppress,
    nullcontext,
    ExitStack,
    asynccontextmanager,
)
from collections.abc import Generator, Iterator
from typing import Any, Callable

## PART 1 — ITERATOR PROTOCOL
Mental model:
  Iterable  — has __iter__(), returns a FRESH iterator each call.
  Iterator  — has __iter__() (returns self) AND __next__() (produces values).
  Rule: iterators are "spent" — consuming one twice yields nothing the second time.
        Iterables are NOT spent — they create a new iterator each time.

In [ ]:
class CountUp:
    """
    A SEPARABLE iterable — __iter__ returns a fresh CountUpIterator.
    This lets you do nested `for a in obj: for b in obj:` correctly.
    """
    def __init__(self, start: int, stop: int) -> None:
        self.start, self.stop = start, stop

    def __iter__(self) -> "CountUpIterator":
        return CountUpIterator(self.start, self.stop)


class CountUpIterator:
    """
    The actual iterator — __iter__ returns self (the cursor).
    Once exhausted, it's done — you can't rewind it.
    """
    def __init__(self, start: int, stop: int) -> None:
        self.current, self.stop = start, stop

    def __iter__(self) -> "CountUpIterator":
        return self          # iterator's __iter__ returns itself

    def __next__(self) -> int:
        if self.current >= self.stop:
            raise StopIteration
        val = self.current
        self.current += 1
        return val


class InfiniteCounter:
    """Unbounded iterator — only safe when a consumer breaks out early."""
    def __init__(self, start: int = 0, step: int = 1) -> None:
        self.current, self.step = start, step

    def __iter__(self) -> "InfiniteCounter": return self
    def __next__(self) -> int:
        val = self.current
        self.current += self.step
        return val


def demo_iterator_protocol() -> None:
    # Separable iterable: two independent traversals
    cu = CountUp(0, 5)
    result1 = list(cu)
    result2 = list(cu)              # fresh iterator; works!
    assert result1 == result2 == [0,1,2,3,4]

    # Direct __next__ usage (what for-loops do under the hood)
    it = iter(CountUp(1, 4))
    assert next(it) == 1
    assert next(it) == 2
    assert next(it) == 3
    try:
        next(it)                    # StopIteration
        assert False
    except StopIteration:
        pass

    # next() with default — no exception on exhaustion
    exhausted = iter([])
    assert next(exhausted, "DONE") == "DONE"

    # Infinite counter — must break
    ic = InfiniteCounter(0, 2)
    evens = [next(ic) for _ in range(5)]
    assert evens == [0,2,4,6,8]

    # Built-in containers: list/dict/str all support the protocol
    lst = [10, 20, 30]
    lit = iter(lst)
    assert next(lit) == 10
    # GOTCHA: consuming the iterator empties it; the original list is unchanged
    assert list(lst) == [10,20,30]  # lst still intact
    assert list(lit) == [20,30]     # iterator was partially consumed

    print("Part 1 (Iterator protocol): ✓")

## PART 2 — GENERATOR FUNCTIONS
Mental model:
  Generator is an iterator you write as a FUNCTION with yield.
  Between yields, Python SUSPENDS the stack frame (all locals + instruction
  pointer) — the function literally pauses, not returns.
  Advantage: O(1) memory regardless of output size; lazy evaluation;
             natural back-pressure.

In [ ]:
def countdown(n: int) -> Iterator[int]:
    """Simple generator — identical to a hand-written iterator class."""
    while n > 0:
        yield n     # suspend here; resume on next()
        n -= 1


def fibonacci() -> Iterator[int]:
    """INFINITE generator — impossible to represent as a list."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


def read_file_lazily(path: str) -> Iterator[str]:
    """
    Memory-efficient file reading.
    With a regular list: loads entire file into RAM.
    With a generator: ONE line at a time, O(1) memory.
    """
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield line.rstrip()


def pipeline_demo(n: int) -> Iterator[int]:
    """
    Generator pipeline — each step is lazy; no intermediate lists created.
    Mental model: a Unix pipe (cat | grep | awk) but in Python.
    """
    numbers = (x for x in range(n))          # step 1: generate
    evens   = (x for x in numbers if x%2==0) # step 2: filter  — lazy
    squared = (x*x for x in evens)           # step 3: transform — lazy
    return squared


def take(gen: Iterator, n: int) -> list:
    """Materialise the first n items from any (possibly infinite) generator."""
    return [next(gen) for _ in range(n)]


def demo_generators() -> None:
    # countdown
    assert list(countdown(5)) == [5,4,3,2,1]
    # generators are SINGLE PASS — consuming twice yields nothing second time
    gen = countdown(3)
    first  = list(gen)
    second = list(gen)
    assert first == [3,2,1] and second == []   # GOTCHA: exhausted

    # fibonacci
    fib_gen = fibonacci()
    assert take(fib_gen, 8) == [0,1,1,2,3,5,8,13]

    # lazy pipeline — no intermediate list; total stays O(1) memory
    result = list(pipeline_demo(100))
    assert result == [x*x for x in range(100) if x%2==0]

    # Generator = iterator: iter(gen) returns itself
    g = countdown(3)
    assert iter(g) is g      # same object

    # Generators report their state
    import inspect
    g2 = countdown(2)
    assert inspect.getgeneratorstate(g2) == "GEN_CREATED"
    next(g2)
    assert inspect.getgeneratorstate(g2) == "GEN_SUSPENDED"
    next(g2)
    try: next(g2)
    except StopIteration: pass
    assert inspect.getgeneratorstate(g2) == "GEN_CLOSED"

    print("Part 2 (Generators): ✓")

## PART 3 — GENERATOR EXPRESSIONS
Mental model: lazy list comprehension — writes like a comprehension but
  produces a generator (O(1) memory) instead of materialising a list.

In [ ]:
def demo_generator_expressions() -> None:
    # List comprehension — materialises ALL values immediately
    squares_list = [x*x for x in range(10**6)]   # ~8 MB

    # Generator expression — lazy; ONE value at a time
    squares_gen  = (x*x for x in range(10**6))   # ~120 bytes

    # Same result but vastly different memory footprint
    assert sum(squares_list) == sum(squares_gen)

    # Chaining generator expressions = pipeline
    numbers   = range(20)
    processed = sum(x*x for x in numbers if x%2==0)  # filter + square + sum
    assert processed == sum(x*x for x in range(0,20,2))

    # Generator expressions as function arguments (no extra parens needed)
    total = sum(len(word) for word in ["hello","world","python"])
    assert total == 16

    # Nested generator expression
    matrix     = [[1,2,3],[4,5,6],[7,8,9]]
    flat_evens = [v for row in matrix for v in row if v % 2 == 0]
    assert flat_evens == [2,4,6,8]

    print("Part 3 (Generator expressions): ✓")

## PART 4 — yield from
Mental model: yield from subgen does THREE things that for/yield cannot:
  1. Forwards next() values from caller to subgen.
  2. Forwards send() values FROM caller INTO subgen.
  3. Captures subgen's return value as the yield from expression's result.
  This is the basis of async/await (await = yield from __await__()).

In [ ]:
def flatten(nested: Any) -> Iterator:
    """Recursive generator that flattens arbitrary nesting."""
    for item in nested:
        if isinstance(item, (list, tuple)):
            yield from flatten(item)  # delegate to sub-generator
        else:
            yield item


def chain_generators(*gens) -> Iterator:
    """Equivalent to itertools.chain."""
    for gen in gens:
        yield from gen


def subgen(label: str) -> Generator:
    """
    Sub-generator that returns a value via `return`.
    yield from captures it as its result.
    """
    yield f"{label}-1"
    yield f"{label}-2"
    return f"{label}-DONE"    # returned to the delegating generator


def delegating() -> Generator:
    result = yield from subgen("A")    # result = "A-DONE"
    yield f"subgen returned: {result}"


def demo_yield_from() -> None:
    # flatten nested structure
    assert list(flatten([1,[2,[3,4]],5])) == [1,2,3,4,5]
    assert list(flatten((1,(2,3),4)))      == [1,2,3,4]

    # chain_generators
    combined = list(chain_generators([1,2], [3,4], [5]))
    assert combined == [1,2,3,4,5]

    # delegation + return value capture
    result = list(delegating())
    assert result == ["A-1","A-2","subgen returned: A-DONE"]

    print("Part 4 (yield from): ✓")

## PART 5 — send() / throw() / close()
Mental model: generators are TWO-WAY channels.
  next(gen)     → pull value from generator  (send None)
  gen.send(val) → inject value into generator, receive next yield
  gen.throw(exc)→ inject exception at current yield point
  gen.close()   → inject GeneratorExit at current yield point

This is the foundation of coroutines and async/await.

In [ ]:
def averaging() -> Generator[float, float, str]:
    """
    Coroutine-style generator.
    Type: Generator[YieldType, SendType, ReturnType]
    Caller:
      gen = averaging()
      next(gen)        # prime (advance to first yield)
      gen.send(10.0)   # injects 10.0, receives running average
      gen.send(20.0)   # injects 20.0, receives new average
      gen.send(None)   # ends the generator
    """
    total = count = 0
    running_avg = 0.0
    while True:
        value = yield running_avg       # suspend; receive via send()
        if value is None:
            return f"final: avg of {count} values = {running_avg:.2f}"
        total += value
        count += 1
        running_avg = total / count


def resilient_gen() -> Generator[str, Any, None]:
    """Handles throw() gracefully."""
    while True:
        try:
            yield "running"
        except ValueError as e:
            yield f"caught: {e}"


def demo_send_throw() -> None:
    # averaging coroutine
    gen = averaging()
    next(gen)                       # prime the generator (advance to first yield)
    gen.send(10.0)                  # running_avg = 10.0
    gen.send(20.0)                  # running_avg = 15.0
    avg = gen.send(30.0)            # running_avg = 20.0
    assert abs(avg - 20.0) < 1e-9

    # close() — injects GeneratorExit
    gen2 = countdown(100)
    next(gen2)
    gen2.close()                    # generator cleaned up
    # Trying to next() after close raises StopIteration
    try:
        next(gen2)
    except StopIteration:
        pass

    # throw() — inject exception at current yield point
    rg = resilient_gen()
    assert next(rg) == "running"
    caught = rg.throw(ValueError, "oops")
    assert caught == "caught: oops"
    assert next(rg) == "running"   # continues normally after handling

    print("Part 5 (send/throw/close): ✓")

## PART 6 — itertools (COMPLETE)
Mental model: lazy building blocks for combinatorial iteration.
  "itertools" is a toolkit of infinite, finite, and combinatoric iterators —
  all lazy, all composable. Avoid materialising unless necessary.

In [ ]:
def demo_itertools() -> None:
    import itertools as it

    # ── INFINITE ITERATORS ────────────────────────────────────────────────────
    # count(start, step): 0, 1, 2, ... / 10, 12, 14, ...
    assert list(it.islice(it.count(10, 2), 5)) == [10,12,14,16,18]

    # cycle(iterable): A, B, C, A, B, C, ...
    assert list(it.islice(it.cycle("ABC"), 7)) == list("ABCABCA")

    # repeat(val, times): [42, 42, 42]
    assert list(it.repeat(42, 3)) == [42,42,42]

    # ── FINITE ITERATORS ──────────────────────────────────────────────────────
    # chain — concatenate iterables lazily
    assert list(it.chain([1,2],[3,4],[5])) == [1,2,3,4,5]
    assert list(it.chain.from_iterable([[1,2],[3,4]])) == [1,2,3,4]

    # compress — keep items where selector is truthy
    assert list(it.compress("ABCDE",[1,0,1,0,1])) == ["A","C","E"]

    # dropwhile / takewhile
    assert list(it.dropwhile(lambda x: x<3, [1,2,3,4,2])) == [3,4,2]
    assert list(it.takewhile(lambda x: x<4, [1,2,3,4,5])) == [1,2,3]

    # filterfalse — keep where predicate is FALSE
    assert list(it.filterfalse(lambda x: x%2, range(8))) == [0,2,4,6]

    # islice — lazy slice (like range but for any iterable)
    assert list(it.islice(range(100), 2, 10, 3)) == [2,5,8]

    # starmap — map over iterable of argument tuples
    assert list(it.starmap(pow, [(2,10),(3,3),(4,2)])) == [1024,27,16]

    # zip_longest — zip with fill for unequal lengths
    pairs = list(it.zip_longest([1,2,3],[4,5], fillvalue=0))
    assert pairs == [(1,4),(2,5),(3,0)]

    # accumulate — running totals (generalised prefix sums)
    import operator
    assert list(it.accumulate([1,2,3,4,5]))              == [1,3,6,10,15]
    assert list(it.accumulate([1,2,3,4],operator.mul))  == [1,2,6,24]

    # groupby — consecutive groups (like SQL GROUP BY on sorted data)
    data = [("A",1),("A",2),("B",3),("B",4),("C",5)]
    groups = {k: list(v) for k, v in it.groupby(data, key=lambda x: x[0])}
    assert groups["A"] == [("A",1),("A",2)]

    # pairwise (3.10+) — adjacent pairs: (1,2),(2,3),(3,4)
    try:
        assert list(it.pairwise([1,2,3,4])) == [(1,2),(2,3),(3,4)]
    except AttributeError:
        pass  # Python < 3.10

    # batched (3.12+) — fixed-size batches
    try:
        assert list(it.batched([1,2,3,4,5], 2)) == [(1,2),(3,4),(5,)]
    except AttributeError:
        pass  # Python < 3.12

    # ── COMBINATORIC ITERATORS ────────────────────────────────────────────────
    # product — Cartesian product
    assert list(it.product("AB","12")) == [("A","1"),("A","2"),("B","1"),("B","2")]
    assert list(it.product(range(2), repeat=3)) == [
        (0,0,0),(0,0,1),(0,1,0),(0,1,1),(1,0,0),(1,0,1),(1,1,0),(1,1,1)]

    # permutations — ordered arrangements
    assert len(list(it.permutations("ABCD", 2))) == 12    # 4*3 = 12

    # combinations — unordered selections (no repeats)
    assert list(it.combinations("ABC",2)) == [("A","B"),("A","C"),("B","C")]

    # combinations_with_replacement — unordered selections WITH repeats
    assert list(it.combinations_with_replacement("AB",2)) == [("A","A"),("A","B"),("B","B")]

    print("Part 6 (itertools): ✓")

## PART 7 — CONTEXT MANAGERS: __enter__ / __exit__ CLASS PROTOCOL
Mental model: a BRACKET around code that guarantees setup + teardown,
  even on exceptions. "The manager owns the resource lifecycle."

__exit__(self, exc_type, exc_val, tb) → bool
  return False (or None) → re-raise any exception
  return True             → SUPPRESS the exception (rarely correct!)


### 🧠 Mental Model: Context Managers

**WHY** — Resources (files, DB connections, locks, network sockets) must be released even when exceptions occur. Without context managers, you'd need try/finally everywhere.

**WHAT** — The `with` statement is guaranteed resource management. It calls `__enter__` before the block and `__exit__` after, no matter what.

**HOW — the protocol:**
```
with EXPR as VAR:
    BODY

# Equivalent to:
_mgr = EXPR
VAR = _mgr.__enter__()
try:
    BODY
except:
    if not _mgr.__exit__(*sys.exc_info()):
        raise          # re-raise if __exit__ returns falsy
else:
    _mgr.__exit__(None, None, None)

__exit__(self, exc_type, exc_val, traceback):
  → Returns truthy   : SUPPRESS the exception (dangerous — only for suppress())
  → Returns falsy    : RE-RAISE the exception (correct default)
```

**Three ways to write a context manager:**

| Method | When to use |
|--------|------------|
| Class with `__enter__`/`__exit__` | Complex setup/teardown with multiple steps |
| `@contextmanager` + `yield` | Simple, single-resource pattern |
| `contextlib.ExitStack` | Dynamic number of context managers |

**`@contextmanager` pattern:**
```python
from contextlib import contextmanager

@contextmanager
def managed_resource():
    resource = acquire()        # __enter__ equivalent
    try:
        yield resource          # body of the `with` block runs here
    finally:
        release(resource)       # __exit__ equivalent (always runs)
```

**`ExitStack` — when you don't know N ahead of time:**
```python
with ExitStack() as stack:
    files = [stack.enter_context(open(f)) for f in file_paths]
    # All files closed when stack exits, even if one fails
```

**WHEN to use context managers:**
- File I/O (always — use `with open()`)
- Database transactions (commit/rollback)
- Threading locks (`with lock:`)
- Timing code blocks (custom timer)
- Temporary directory/config changes


In [ ]:
class Timer:
    """Measure elapsed time of a with-block."""
    def __enter__(self) -> "Timer":
        self.start   = time.perf_counter()
        self.elapsed = 0.0
        return self                        # bound to `as t`

    def __exit__(self, *_: Any) -> None:
        self.elapsed = time.perf_counter() - self.start
        # returning None (falsy) → never suppress exceptions


class Transaction:
    """Database transaction — COMMIT on success, ROLLBACK on exception."""
    def __init__(self, log: list[str]) -> None: self.log = log

    def __enter__(self) -> "Transaction":
        self.log.append("BEGIN"); return self

    def execute(self, sql: str) -> None:
        self.log.append(f"EXEC: {sql}")

    def __exit__(self, exc_type, exc, tb) -> bool:
        self.log.append("ROLLBACK" if exc_type else "COMMIT")
        return False           # NEVER suppress — let the exception propagate


class ManagedFile:
    """
    Custom file context manager showing three phases:
      __enter__: acquire resource
      body:      use resource
      __exit__:  release resource
    """
    def __init__(self, path: str, mode: str = "r") -> None:
        self.path, self.mode = path, mode
        self._file = None

    def __enter__(self):
        self._file = open(self.path, self.mode, encoding="utf-8")
        return self._file

    def __exit__(self, exc_type, exc, tb) -> bool:
        if self._file:
            self._file.close()
        return False


def demo_context_manager_class() -> None:
    # Timer
    with Timer() as t:
        _ = sum(range(100_000))
    assert t.elapsed >= 0

    # Transaction success
    log: list[str] = []
    with Transaction(log) as tx:
        tx.execute("INSERT INTO orders VALUES (1)")
    assert log == ["BEGIN","EXEC: INSERT INTO orders VALUES (1)","COMMIT"]

    # Transaction failure — ROLLBACK runs, exception propagates
    log2: list[str] = []
    try:
        with Transaction(log2) as tx2:
            tx2.execute("INSERT ...")
            raise ValueError("constraint violation")
    except ValueError:
        pass
    assert log2[-1] == "ROLLBACK"

    print("Part 7 (Context manager class): ✓")

## PART 8 — @contextmanager
Mental model: write a generator that yields ONCE.
  Code BEFORE yield = __enter__
  yield value        = the `as` target
  Code AFTER yield   = __exit__ (in finally for exception safety)

In [ ]:
@contextmanager
def timer_gen():
    """Same as Timer class above — one line shorter."""
    start = time.perf_counter()
    try:
        yield                          # execution enters the with-body here
    finally:
        elapsed = time.perf_counter() - start
        # NOTE: finally runs even if the body raised an exception


@contextmanager
def db_transaction(log: list[str]):
    log.append("BEGIN")
    try:
        yield log
        log.append("COMMIT")
    except Exception:
        log.append("ROLLBACK")
        raise                          # re-raise — do NOT swallow


@contextmanager
def temporary_directory():
    """Creates a temp dir; deletes it on exit."""
    import tempfile, shutil
    path = tempfile.mkdtemp()
    try:
        yield path
    finally:
        shutil.rmtree(path, ignore_errors=True)


@contextmanager
def patched(obj, attr: str, value: Any):
    """
    Temporarily set obj.attr = value; restore on exit.
    Classic use: mocking in tests without a full mock library.
    """
    old = getattr(obj, attr, None)
    setattr(obj, attr, value)
    try:
        yield
    finally:
        if old is None:
            try: delattr(obj, attr)
            except AttributeError: pass
        else:
            setattr(obj, attr, old)


def demo_contextmanager() -> None:
    with timer_gen():
        _ = sum(range(100_000))

    # db_transaction success
    log: list[str] = []
    with db_transaction(log) as tx:
        tx.append("SELECT")
    assert log == ["BEGIN","SELECT","COMMIT"]

    # db_transaction failure — ROLLBACK + re-raise
    log2: list[str] = []
    try:
        with db_transaction(log2):
            log2.append("INSERT")
            raise RuntimeError("oops")
    except RuntimeError:
        pass
    assert "ROLLBACK" in log2

    # temporary_directory
    with temporary_directory() as d:
        import os; assert os.path.isdir(d)
    assert not os.path.exists(d)    # cleaned up

    # patched — monkey-patching without side effects
    class Config:
        debug = False

    with patched(Config, "debug", True):
        assert Config.debug is True
    assert Config.debug is False    # restored

    print("Part 8 (@contextmanager): ✓")

## PART 9 — contextlib UTILITIES

In [ ]:
def demo_contextlib() -> None:
    # suppress — swallow specific exceptions elegantly
    with suppress(FileNotFoundError):
        open("/no/such/file/exists.txt")   # would normally raise
    # execution continues here — the exception was suppressed

    # nullcontext — placeholder context manager (does nothing)
    # Use Case: conditional context managers
    def process(data, lock=None):
        with (lock if lock is not None else nullcontext()):
            return data  # lock is optional

    assert process([1,2,3]) == [1,2,3]

    # ExitStack — manage a dynamic list of context managers
    # Use Case: open N files, where N is known only at runtime
    import io
    buffers = [io.StringIO() for _ in range(3)]
    with ExitStack() as stack:
        opened = [stack.enter_context(buf) for buf in buffers]
        for i, f in enumerate(opened):
            f.write(f"data{i}")
    # all StringIO objects closed when ExitStack exits

    # ExitStack as cleanup registry
    cleanup_log: list[str] = []
    with ExitStack() as stack:
        stack.callback(cleanup_log.append, "cleanup-1")
        stack.callback(cleanup_log.append, "cleanup-2")
    assert cleanup_log == ["cleanup-2","cleanup-1"]  # LIFO — last registered, first called

    print("Part 9 (contextlib utilities): ✓")

## PART 10 — ASYNC GENERATORS & ASYNC CONTEXT MANAGERS (preview)
Mental model: async generators suspend at `yield` points while ALSO being
  suspendable by the event loop at `await` points.
  Used with `async for` and `async with` in coroutines.

In [ ]:
@asynccontextmanager
async def async_db_connection(url: str):
    """async with: acquire resource asynchronously."""
    # In real code: conn = await asyncpg.connect(url)
    conn = {"url": url, "open": True}
    try:
        yield conn
    finally:
        conn["open"] = False  # await conn.close() in real code


async def async_range(n: int):
    """Async generator — yields values with optional async between yields."""
    for i in range(n):
        # await asyncio.sleep(0) would yield control to event loop here
        yield i


async def demo_async_generators() -> None:
    """Run with: asyncio.run(demo_async_generators())"""
    import asyncio

    # async context manager
    async with async_db_connection("postgres://localhost/mydb") as conn:
        assert conn["open"] is True
    assert conn["open"] is False    # cleanup ran

    # async for loop
    result: list[int] = []
    async for val in async_range(5):
        result.append(val)
    assert result == [0,1,2,3,4]


def demo_async_preview() -> None:
    """Execute the async demo synchronously via asyncio.run()."""
    import asyncio
    asyncio.run(demo_async_generators())
    print("Part 10 (async generators/context managers): ✓")

## MAIN

In [ ]:
def main() -> None:
    print("=" * 70)
    print("ITERATORS / GENERATORS / CONTEXT MANAGERS — complete reference")
    print("=" * 70)
    demo_iterator_protocol()
    demo_generators()
    demo_generator_expressions()
    demo_yield_from()
    demo_send_throw()
    demo_itertools()
    demo_context_manager_class()
    demo_contextmanager()
    demo_contextlib()
    demo_async_preview()
    print("-" * 70)
    print("All iterator/generator/context manager demos passed ✔")


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()